In [1]:
import langextract as lx
import textwrap
from docling.document_converter import DocumentConverter
import os
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# IMPORTANTE: Verificar se você tem uma API key da OpenAI
# Se você só tem a chave do Groq, obtenha uma da OpenAI em:
# https://platform.openai.com/api-keys

api_key = os.getenv("OPENAI_API_KEY_2")
if not api_key:
    print("❌ ERRO: OPENAI_API_KEY não encontrada no .env!")
    print("   Adicione no seu .env: OPENAI_API_KEY=sk-sua-chave-aqui")
    raise ValueError("API key da OpenAI não configurada")

if not api_key.startswith("sk-"):
    print(f"❌ ERRO: Esta não é uma API key da OpenAI (começa com '{api_key[:3]}...')")
    print("   O langextract precisa da API key da OpenAI (sk-...), não do Groq (gsk-...)")
    print("   Obtenha uma em: https://platform.openai.com/api-keys")
    raise ValueError("API key inválida")

print(f"✅ API Key da OpenAI configurada: {api_key[:10]}...{api_key[-4:]}")

# Converter o PDF
converter = DocumentConverter()
result = converter.convert("./2408.09869v5.pdf")
markdown_output = result.document.export_to_markdown()

first_pages = markdown_output[:6000]
prompt = textwrap.dedent("""\
Extract metadata from this technical report including title, all authors, 
affiliation, version number, and GitHub repository URLs.
Use exact text from the document.
""")

/Users/luizfelipew/Documents/git/AI-Engineering/dev-eficiente-IA/engineering-ai/curso-ia/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-26 22:11:14,173 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-26 22:11:14,312 - INFO - Going to convert document batch...
2025-11-26 22:11:14,313 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-26 22:11:14,341 - INFO - Loading plugin 'docling_defaults'
2025-11-26 22:11:14,344 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-26 22:11:14,361 - INFO - Loading plugin 'docling_defaults'
2025-11-26 22:11:14,367 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']


✅ API Key da OpenAI configurada: sk-proj-lj...em4A


2025-11-26 22:11:15,320 - INFO - Auto OCR model selected ocrmac.
2025-11-26 22:11:15,328 - INFO - Accelerator device: 'mps'
2025-11-26 22:11:17,191 - INFO - Accelerator device: 'mps'
2025-11-26 22:11:17,782 - INFO - Processing document 2408.09869v5.pdf
2025-11-26 22:11:26,923 - INFO - Finished converting document 2408.09869v5.pdf in 12.75 sec.


In [2]:
examples = [
    lx.data.ExampleData(
        text="Docling Technical Report\nVersion 1.0\nChristoph Auer Maksym Lysak Ahmed Nassar\nAI4K Group, IBM Research\nRüschlikon, Switzerland\ngithub.com/DS4SD/docling",
        extractions=[
            lx.data.Extraction(
                extraction_class="title",
                extraction_text="Docling Technical Report",
                attributes={},
            ),
            lx.data.Extraction(
                extraction_class="author",
                extraction_text="Christoph Auer",
                attributes={},
            ),
            lx.data.Extraction(
                extraction_class="author", extraction_text="Maksym Lysak", attributes={}
            ),
            lx.data.Extraction(
                extraction_class="affiliation",
                extraction_text="AI4K Group, IBM Research",
                attributes={},
            ),
            lx.data.Extraction(
                extraction_class="url",
                extraction_text="github.com/DS4SD/docling",
                attributes={"type": "repository"},
            ),
        ],
    )
]


In [3]:
# Executar a extração de metadados
print("🔄 Executando extração de metadados...")

extraction_result = lx.extract(
    text_or_documents=first_pages,
    prompt_description=prompt,
    examples=examples,
    model_id="gpt-4o-mini",
    api_key=api_key,  # Passar a API key explicitamente
)

print("✅ Extração concluída!")

🔄 Executando extração de metadados...


2025-11-26 22:11:29,119 - INFO - Loaded provider plugin: gemini
2025-11-26 22:11:29,122 - INFO - Loaded provider plugin: ollama
2025-11-26 22:11:29,124 - INFO - Loaded provider plugin: openai
LangExtract: model=gpt-4o-mini, current=5,994 chars, processed=0 chars:  [00:00]2025-11-26 22:11:31,816 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-26 22:11:32,226 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-26 22:11:32,741 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-26 22:11:32,749 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-26 22:11:34,275 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-26 22:11:34,888 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-26 22:11:43,903 - INFO - HTTP Request: POS

✅ Extração concluída!


In [4]:
lx.io.save_annotated_documents(
    [extraction_result], output_name="docling_paper_metadata.json"
)

LangExtract: Saving to docling_paper_metadata.json: 1 docs [00:00, 673.35 docs/s]

✓ Saved 1 documents to docling_paper_metadata.json


In [5]:
print("-" * 80)
for extraction in extraction_result.extractions:
    print(f"{extraction.extraction_class}: {extraction.extraction_text}")
    if extraction.attributes:
        print(f"  Atributos: {extraction.attributes}")


--------------------------------------------------------------------------------
title: Docling Technical Report
author: Christoph Auer
author: Maksym Lysak
author: Ahmed Nassar
author: Michele Dolfi
author: Nikolaos Livathinos
author: Panos Vagenas
author: Cesar Berrospi Ramis
author: Matteo Omenetti
author: Fabian Lindlbauer
author: Kasper Dinkla
author: Lokesh Mishra
author: Yusik Kim
author: Shubham Gupta
author: Rafael Teixeira de Lima
author: Valery Weber
author: Lucas Morin
author: Ingmar Meijer
author: Viktor Kuropiatnyk
author: Peter W. J. Staar
affiliation: AI4K Group, IBM Research
affiliation: Rüschlikon, Switzerland
version: Version 1.0
url: github.com/DS4SD/docling
  Atributos: {'type': 'repository'}
title: Docling Technical Report
author: Christoph Auer
author: Maksym Lysak
author: Ahmed Nassar
affiliation: AI4K Group, IBM Research
version: 1.0
url: github.com/DS4SD/docling
  Atributos: {'type': 'repository'}
title: DocLayNet: A Large Human-Annotated Dataset for Document-